# RoofTop Regression Analysis

## Purpose

This notebook investigates a **regression in rooftop metrics** for APT (Address Point) data.
A regression here means rooftop accuracy has dropped between two snapshots, and the most
likely cause is that some APTs present in the **old snapshot** are **missing from the latest
snapshot**.

## Approach

The first step is to identify **which APTs went missing** between the two snapshots:

1. **Load the old snapshot** into a Delta table.
2. **Load the latest snapshot** into a Delta table.
3. **Find the missing APTs** — the APTs present in the old snapshot but absent from the
   latest snapshot (a `leftanti` join on the APT `unsigned_id`).

The set of missing APTs is the input for the downstream rooftop regression investigation.


In [0]:
%sql
-- Create the database used to store the snapshot Delta tables.
-- Run this once before the snapshot loader cells.
CREATE DATABASE IF NOT EXISTS rooftop_regression_db;


In [0]:
%scala
// ============================================================
// Load OLD snapshot -> Delta table
// Pin OLD_REVISION to the older revision you want to compare against.
// Saved to: rooftop_regression_db.apt_snapshot_revision_{OLD_REVISION}_14533
// ============================================================

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import org.apache.sedona.spark.SedonaContext
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.orbis.addressing.bulk.commons.odp.LoadSnapshotInfo
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository

val LAYER_ID = 14533

// >>> Set the OLD snapshot revision id here <<<
val OLD_REVISION = 42372254L

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
println(s"Environment Name $env")

implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)
println(ConfigLoader.getConfig.getString("secrets.key-vault.url"))

println(s"Loading OLD snapshot revision: $OLD_REVISION")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(LAYER_ID, OLD_REVISION)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

val config = SourceConfigLoader.getConfig
val deltaConfig = config.getTableMapping.getOrDefault(env, "preprocess_dev.source_data")
val Array(databaseName, tableName) = deltaConfig.split("\\.")
println(s"Database Name: $databaseName")
println(s"Table Name: $tableName")

SparkHelper.init("rooftop_regression_db")
new LoadFreshSnapshotData().run()

// Persist the loaded old snapshot into a Delta table
val oldSnapshotTable = s"rooftop_regression_db.apt_snapshot_revision_${OLD_REVISION}_${LAYER_ID}"
val oldAptDS = new OrbisElementRepository(LAYER_ID.toString).readAll
oldAptDS.write.format("delta").mode("overwrite").saveAsTable(oldSnapshotTable)
println(s"OLD snapshot written to: $oldSnapshotTable")
display(oldAptDS)


In [0]:
%scala
// ============================================================
// Load LATEST snapshot -> Delta table
// Uses LoadSnapshotInfo.getLatestRevisionId to resolve the newest revision.
// Saved to: rooftop_regression_db.apt_snapshot_revision_{latestRevision}_14533
// ============================================================

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import org.apache.sedona.spark.SedonaContext
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.orbis.addressing.bulk.commons.odp.LoadSnapshotInfo
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository

val LAYER_ID = 14533

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
println(s"Environment Name $env")

implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)
println(ConfigLoader.getConfig.getString("secrets.key-vault.url"))

val latestRevision = LoadSnapshotInfo.getLatestRevisionId(LAYER_ID)
println(s"Loading LATEST snapshot revision: ${latestRevision.get}")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(LAYER_ID, latestRevision.get)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

val config = SourceConfigLoader.getConfig
val deltaConfig = config.getTableMapping.getOrDefault(env, "preprocess_dev.source_data")
val Array(databaseName, tableName) = deltaConfig.split("\\.")
println(s"Database Name: $databaseName")
println(s"Table Name: $tableName")

SparkHelper.init("rooftop_regression_db")
new LoadFreshSnapshotData().run()

// Persist the loaded latest snapshot into a Delta table
val latestSnapshotTable = s"rooftop_regression_db.apt_snapshot_revision_${latestRevision.get}_${LAYER_ID}"
val latestAptDS = new OrbisElementRepository(LAYER_ID.toString).readAll
latestAptDS.write.format("delta").mode("overwrite").saveAsTable(latestSnapshotTable)
println(s"LATEST snapshot written to: $latestSnapshotTable")
display(latestAptDS)


In [0]:
%scala
// ============================================================
// Read OLD snapshot Delta table as Dataset[OrbisElement]
// then enrich with `country` and `orbisIdString` columns.
// ============================================================

import org.apache.spark.sql.{Dataset, Encoders}
import org.apache.spark.sql.functions._
import com.tomtom.orbis.io.spark.model.{Id, OrbisElement}
import java.lang.{Long => JLong}

// Build the colon-separated unsigned id string: {layerId}:{unsignedHigh}:{unsignedLow}
def convertOrbisIdToString(orbisId: Id): String = {
  val COLON_SEPARATOR = ":"
  Seq(orbisId.layerId.getOrElse(14533).toString,
      JLong.toUnsignedString(orbisId.high),
      JLong.toUnsignedString(orbisId.low)).mkString(COLON_SEPARATOR)
}
val convertOrbisIdUDF = udf((orbisId: Id) => convertOrbisIdToString(orbisId))

val countryTagKey = "metadata:country"

val oldSnapshotTable = "rooftop_regression_db.apt_snapshot_revision_42372254_14533"

val oldSnapshotDS: Dataset[OrbisElement] =
  spark.table(oldSnapshotTable).as[OrbisElement](Encoders.product[OrbisElement])

val oldSnapshotEnriched = oldSnapshotDS
  // country = value of the metadata:country tag
  .withColumn("country",
    filter(col("tags"), t => t.getField("tagKey").getField("key") === countryTagKey)
      .getItem(0).getField("value"))
  // orbisIdString = unsigned colon-separated id
  .withColumn("orbisIdString", convertOrbisIdUDF(col("id")))

println(s"OLD snapshot read from: $oldSnapshotTable, count: ${oldSnapshotEnriched.count()}")
display(oldSnapshotEnriched)


In [0]:
%scala
// ============================================================
// Read LATEST snapshot Delta table as Dataset[OrbisElement]
// then enrich with `country` and `orbisIdString` columns.
// (convertOrbisIdUDF and countryTagKey are defined in the OLD-read cell.)
// ============================================================

import org.apache.spark.sql.{Dataset, Encoders}
import org.apache.spark.sql.functions._
import com.tomtom.orbis.io.spark.model.OrbisElement

val latestSnapshotTable = "rooftop_regression_db.apt_snapshot_revision_42615928_14533"

val latestSnapshotDS: Dataset[OrbisElement] =
  spark.table(latestSnapshotTable).as[OrbisElement](Encoders.product[OrbisElement])

val latestSnapshotEnriched = latestSnapshotDS
  // country = value of the metadata:country tag
  .withColumn("country",
    filter(col("tags"), t => t.getField("tagKey").getField("key") === countryTagKey)
      .getItem(0).getField("value"))
  // orbisIdString = unsigned colon-separated id
  .withColumn("orbisIdString", convertOrbisIdUDF(col("id")))

println(s"LATEST snapshot read from: $latestSnapshotTable, count: ${latestSnapshotEnriched.count()}")
display(latestSnapshotEnriched)


In [ ]:
%scala
// ============================================================
// Count APTs that carry the metadata:apa:improvement = yes tag
// AND belong to country ESP, in both the OLD and LATEST snapshots.
// ============================================================

import org.apache.spark.sql.functions._

val COUNTRY_ISO = "ESP"

def apaImprovementCount(ds: org.apache.spark.sql.DataFrame): Long =
  ds.filter(exists(col("tags"), t =>
      t.getField("tagKey").getField("key") === "metadata:apa:improvement"
      && t.getField("value") === "yes"
    ))
    .filter(col("country") === COUNTRY_ISO)
    .count()

val oldApaImprovementCount = apaImprovementCount(oldSnapshotEnriched)
val latestApaImprovementCount = apaImprovementCount(latestSnapshotEnriched)

println(s"OLD snapshot   - metadata:apa:improvement=yes & country=$COUNTRY_ISO count: $oldApaImprovementCount")
println(s"LATEST snapshot - metadata:apa:improvement=yes & country=$COUNTRY_ISO count: $latestApaImprovementCount")


In [ ]:
%scala
// ============================================================
// Inspect specific APT ids: location (lat/lng/wkt) + tags
// in both the OLD and LATEST snapshots.
// (oldMatches / latestMatches are displayed in the cells below.)
// ============================================================

import org.apache.spark.sql.functions._

val targetIds = Seq(
  "14533:11516613567023742976:6462522384025228140",
  "14533:11516615605049950208:12337090556445643619",
  "14533:11516612161269989376:12408683332341800902",
  "14533:11516612091690156032:8657352373094294334"
)

val inspectCols = Seq(col("orbisIdString"), col("country"), col("lat"), col("lng"), col("wkt"), col("tags"))

val oldMatches = oldSnapshotEnriched.filter(col("orbisIdString").isin(targetIds: _*)).select(inspectCols: _*)
val latestMatches = latestSnapshotEnriched.filter(col("orbisIdString").isin(targetIds: _*)).select(inspectCols: _*)

println(s"found ${oldMatches.count()} of ${targetIds.size} ids in OLD snapshot")
println(s"found ${latestMatches.count()} of ${targetIds.size} ids in LATEST snapshot")


In [ ]:
%scala
// ===== OLD snapshot =====
display(oldMatches)


In [ ]:
%scala
// ===== LATEST snapshot =====
display(latestMatches)


In [ ]:
%scala
// ============================================================
// Join OLD vs LATEST matches on orbisIdString.
// Keep ALL rows from oldMatches (left join); latest_* columns
// will be null where there is no matching LATEST row.
// ============================================================

import org.apache.spark.sql.functions._

val oldRenamed = oldMatches
  .withColumnRenamed("lat", "old_lat")
  .withColumnRenamed("lng", "old_lon")
  .withColumnRenamed("wkt", "old_wkt")
  .withColumnRenamed("tags", "old_tags")
  .drop("country")

val latestRenamed = latestMatches
  .withColumnRenamed("lat", "latest_lat")
  .withColumnRenamed("lng", "latest_lon")
  .withColumnRenamed("wkt", "latest_wkt")
  .withColumnRenamed("tags", "latest_tags")
  .drop("country")

val oldVsLatest = oldRenamed
  .join(latestRenamed, Seq("orbisIdString"), "left")
  .select(
    col("orbisIdString"),
    col("old_lat"), col("old_lon"), col("old_wkt"), col("old_tags"),
    col("latest_lat"), col("latest_lon"), col("latest_wkt"), col("latest_tags")
  )

display(oldVsLatest)
